# PIMA Diabetes — QML Comparison (PennyLane)
This notebook compares, under the **same preprocessing + cross-validation protocol**:

**(A)** AngleEmbedding Variational Quantum Classifier (VQC)  
**(B)** Hamiltonian-Embedding VQC (data Hamiltonian + time evolution)  
**(C)** Hamiltonian Quantum Kernel (fidelity kernel + SVM)

**Protocol**
- KaggleHub download (as requested)
- Stratified K-Fold cross-validation (default: 5 folds)
- Per-fold preprocessing fit only on train split: impute → scale → PCA-to-qubits → angle mapping
- Metrics per fold: ROC AUC, PR AUC, Accuracy@0.5, F1@0.5  
- Summary table with mean ± std

> Tip: VQC training is the slowest part. If runtime is high, reduce `EPOCHS`, `N_FOLDS`, or `n_qubits`.


* **01 Comparison**:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Megaantony00/qml-diabetes-nisq/blob/main/notebooks/01_qml_comparison_angle_ham_kernel_cv.ipynb)

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q pennylane>=0.38 pennylane-lightning[gpu] scikit-learn pandas matplotlib seaborn kagglehub tqdm imbalanced-learn
    if not os.path.exists('results'): os.makedirs('results')
    print("Environment configured for Colab.")
else:
    print("Environment configured for Local/Server usage.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 54.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

# 1. Monta il Drive
drive.mount('/content/drive', force_remount=True)

# 2. IMPOSTA QUI I QUBIT (Cambia da 6 a 8 per la seconda run)
n_qubits = 8
n_components = n_qubits # La PCA userà questo numero

# 3. CREA IL PERCORSO DINAMICO
# In questo modo crea cartelle diverse per test diversi
drive_path = f"/content/drive/MyDrive/Tesi_Risultati_{n_qubits}Qubits_Noise_CV/"

if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"✅ Nuova cartella creata: {drive_path}")
else:
    print(f"✅ Cartella esistente: {drive_path}")

print(f"🚀 Pronto per la run a {n_qubits} Qubit.")

Mounted at /content/drive
✅ Cartella esistente: /content/drive/MyDrive/Tesi_Risultati_8Qubits_Noise_CV/
🚀 Pronto per la run a 8 Qubit.


## 1) Imports

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score
)

import pennylane as qml
from tqdm.auto import tqdm

np.random.seed(42)

## 2) Download dataset (KaggleHub)

In [ ]:
import kagglehub

# Download latest version (as provided)
path = kagglehub.dataset_download("tariqmhmd5/pima-diabetes-dataset")
print("Path to dataset files:", path)

# Find a CSV inside the downloaded folder
csv_candidates = glob.glob(str(Path(path) / "**" / "*.csv"), recursive=True)
print("CSV candidates:", csv_candidates)

if not csv_candidates:
    raise FileNotFoundError("No CSV file found in the KaggleHub dataset folder.")
csv_path = csv_candidates[0]
print("Using:", csv_path)

100%|██████████| 9.16k/9.16k [00:00<00:00, 13.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/tariqmhmd5/pima-diabetes-dataset/versions/1
CSV candidates: ['/root/.cache/kagglehub/datasets/tariqmhmd5/pima-diabetes-dataset/versions/1/Diabetes.csv']
Using: /root/.cache/kagglehub/datasets/tariqmhmd5/pima-diabetes-dataset/versions/1/Diabetes.csv


## 3) Load data

In [ ]:
df = pd.read_csv(csv_path)
display(df.head())
print("Shape:", df.shape)

# Identifica la colonna dell'esito
label_col = "Outcome" if "Outcome" in df.columns else df.columns[-1]
feature_cols = [c for c in df.columns if c != label_col]

X = df[feature_cols].copy()

# Mappatura delle stringhe in numeri (gestisce YES/NO, Positive/Negative o stringhe '1'/'0')
y = df[label_col].map({
    'YES': 1, 'NO': 0,
    'positive': 1, 'negative': 0,
    '1': 1, '0': 0,
    1: 1, 0: 0
}).astype(int).values

# Sostituzione degli zeri con NaN dove indicano valori mancanti (tipico di PIMA)
zero_as_missing = [c for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"] if c in X.columns]
X[zero_as_missing] = X[zero_as_missing].replace(0, np.nan)

X = X.values.astype(float)
print("X shape:", X.shape, "y mean (percentuale positivi):", y.mean())

,Number of times pregnant,Plasma glucose concentration,Diastolic blood pressure,Triceps skin fold thickness,2-Hour serum insulin,Body mass index,Diabetes pedigree function,Age (years),Class variable
0,6,148,72,35,0,33.6,0.627,50,YES
1,1,85,66,29,0,26.6,0.351,31,NO
2,8,183,64,0,0,23.3,0.672,32,YES
3,1,89,66,23,94,28.1,0.167,21,NO
4,0,137,40,35,168,43.1,2.288,33,YES


Shape: (768, 9)
X shape: (768, 8) y mean (percentuale positivi): 0.3489583333333333


## 4) Experiment configuration

In [ ]:
# Cross-validation
N_FOLDS = 5
RANDOM_STATE = 42

# QML capacity
#n_qubits = 8       # PCA components -> qubits
N_LAYERS = 2       # trainable layers in VQC head

# Training (VQC)
EPOCHS = 80
BATCH_SIZE = 48
LR = 0.05

# Hamiltonian embedding / kernel hyperparameters
ALPHA_ZZ = 0.5       # ZZ coupling strength
T_EMBED = 1.0        # evolution time
TROTTER_STEPS = 1    # more steps => better approx but deeper circuits

# Kernel SVM regularization
SVC_C = 1.0

print("Config:",
      f"N_FOLDS={N_FOLDS}, n_qubits={n_qubits}, N_LAYERS={N_LAYERS}, EPOCHS={EPOCHS}")

Config: N_FOLDS=5, n_qubits=8, N_LAYERS=2, EPOCHS=80


## 5) Shared preprocessing (fit on train split only)
We fit per fold:
- Median imputation
- Standard scaling
- PCA to `n_qubits`
- Robust mapping to angles in \([-\pi, \pi]\) using train quantiles (5%–95%)


In [ ]:
def fit_transform_fold(X_train, X_val, n_qubits=n_qubits):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    #pca = PCA(n_components=n_qubits, random_state=RANDOM_STATE)
    pca = PCA(n_components=n_components)

    Xtr = scaler.fit_transform(imputer.fit_transform(X_train))
    Xva = scaler.transform(imputer.transform(X_val))

    Xtr_q = pca.fit_transform(Xtr)
    Xva_q = pca.transform(Xva)

    q_low = np.quantile(Xtr_q, 0.05, axis=0)
    q_high = np.quantile(Xtr_q, 0.95, axis=0)

    def to_angles(Z):
        Zc = np.clip(Z, q_low, q_high)
        Zn = (Zc - q_low) / (q_high - q_low + 1e-9)  # [0,1]
        return (Zn * 2 - 1) * np.pi                  # [-pi, pi]

    return to_angles(Xtr_q), to_angles(Xva_q)

## 6) Circuits: (A) AngleEmbedding VQC and (B) Hamiltonian-Embedding VQC

In [ ]:
import pennylane.numpy as pnp

# 1. Cambio device: default.mixed è necessario per simulare il rumore (CPU)
dev_noisy = qml.device("default.mixed", wires=n_qubits)

# 2. Setup del Modello di Rumore (Noise Model)
# Includiamo tutti i gate usati nel tuo codice
fcond = qml.noise.op_in([qml.RX, qml.RY, qml.RZ, qml.Rot, qml.Hadamard])
noise_after_rot = qml.noise.partial_wires(qml.DepolarizingChannel, 0.01)

meas_cond = qml.noise.meas_eq(qml.expval)
readout_noise = qml.noise.partial_wires(qml.PhaseFlip, 0.02)

noise_model = qml.NoiseModel(
    {fcond: noise_after_rot},
    {meas_cond: readout_noise}
)

# --- FUNZIONI DI SUPPORTO (Mantieni queste definizioni!) ---
def head_ansatz(weights):
    for l in range(N_LAYERS):
        for i in range(n_qubits):
            qml.Rot(weights[l, i, 0], weights[l, i, 1], weights[l, i, 2], wires=i)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i+1) % n_qubits])

def data_hamiltonian(x, alpha=ALPHA_ZZ):
    coeffs = []
    ops = []
    for i in range(n_qubits):
        coeffs.append(x[i])
        ops.append(qml.PauliZ(i))
    for i in range(n_qubits):
        for j in range(i+1, n_qubits):
            coeffs.append(alpha * x[i] * x[j])
            ops.append(qml.PauliZ(i) @ qml.PauliZ(j))
    return qml.Hamiltonian(coeffs, ops)

# --- QNODES RUMOROSI (Annotati con il noise_model) ---
@qml.noise.add_noise(noise_model=noise_model)
@qml.qnode(dev_noisy, interface="autograd")
def vqc_angle(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    head_ansatz(weights)
    return qml.expval(qml.PauliZ(0))

@qml.noise.add_noise(noise_model=noise_model)
@qml.qnode(dev_noisy, interface="autograd")
def vqc_hamiltonian(x, weights):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    Hx = data_hamiltonian(x)
    qml.ApproxTimeEvolution(Hx, T_EMBED, TROTTER_STEPS)
    head_ansatz(weights)
    return qml.expval(qml.PauliZ(0))

# --- FUNZIONI DI TRAINING E LOSS ---
def sigmoid(z):
    return 1.0 / (1.0 + pnp.exp(-z))

def predict_proba_vqc(qnode, X, weights):
    logits = pnp.array([qnode(x, weights) for x in X])
    return sigmoid(logits)

def bce(y_true, p_pred, eps=1e-9):
    p = pnp.clip(p_pred, eps, 1 - eps)
    return -pnp.mean(y_true * pnp.log(p) + (1 - y_true) * pnp.log(1 - p))

### Training helper (shared by A and B)

In [ ]:
def train_vqc(qnode, X_train, y_train, X_val, y_val, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=42):
    # Usiamo pnp per i pesi in modo che siano differenziabili
    rng = np.random.default_rng(seed)

    # Inizializzazione pesi come PennyLane numpy array con gradiente attivo
    raw_weights = 0.01 * rng.standard_normal((N_LAYERS, n_qubits, 3))
    weights = pnp.array(raw_weights, requires_grad=True)

    opt = qml.optimize.AdamOptimizer(stepsize=lr)

    # Ciclo di addestramento mini-batch
    for _ in range(epochs):
        idx = rng.permutation(len(X_train))
        Xs, ys = X_train[idx], y_train[idx]
        for start in range(0, len(Xs), batch_size):
            end = start + batch_size
            Xb = Xs[start:end]
            yb = ys[start:end]

            # L'ottimizzatore aggiorna i pesi basandosi sulla funzione di costo (BCE)
            weights = opt.step(lambda w: bce(yb, predict_proba_vqc(qnode, Xb, w)), weights)

    # Calcolo probabilità finali sul set di validazione
    p_val = predict_proba_vqc(qnode, X_val, weights)
    return weights, p_val

## 7) (C) Hamiltonian Quantum Kernel + SVM

We define the **kernel** as the fidelity between Hamiltonian-embedded states:
\[
K(x, x') = |\langle \phi(x) | \phi(x') \rangle|^2, \quad |\phi(x)\rangle = e^{-iH(x)t}|+\rangle^{\otimes n}
\]

Then train an SVM with `kernel='precomputed'`.

> Computing an \(N\times N\) kernel matrix is **O(N²)**. For PIMA this is usually feasible, but still slower than classical baselines.


In [ ]:
dev_k_noisy = qml.device("default.mixed", wires=n_qubits)

@qml.noise.add_noise(noise_model=noise_model)
@qml.qnode(dev_k_noisy, interface="autograd")
def ham_feature_state(x):
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    Hx = data_hamiltonian(x)
    qml.ApproxTimeEvolution(Hx, T_EMBED, TROTTER_STEPS)
    # Fondamentale: restituisce la density matrix, non lo state
    return qml.density_matrix(wires=range(n_qubits))

def kernel_matrix(A, B):
    """Versione OTTIMIZZATA per Matrici di Densità (Rumore)"""
    print(f"   (Calcolo stati rumorosi per {len(A) if A is B else len(A)+len(B)} campioni...)")

    states_A = [ham_feature_state(x) for x in A]
    states_B = states_A if A is B else [ham_feature_state(x) for x in B]

    K = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            # Hilbert-Schmidt Inner Product per matrici di densità: Tr(rho_A * rho_B)
            K[i, j] = np.trace(states_A[i] @ states_B[j]).real
    return K

## 8) Cross-validation loop (same folds, same preprocessing)

In [ ]:
import os
import pickle
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

# --- 1. CONFIGURAZIONE CHECKPOINT ---
# Il file verrà salvato direttamente nella tua cartella su Drive
checkpoint_file = os.path.join(drive_path, f"cv_checkpoint_{n_qubits}qubits.pkl")

def fold_metrics_from_probs(y_true, p):
    auc = roc_auc_score(y_true, p)
    ap = average_precision_score(y_true, p)
    yhat = (p >= 0.5).astype(int)
    acc = accuracy_score(y_true, yhat)
    f1 = f1_score(y_true, yhat)
    return {"roc_auc": auc, "pr_auc": ap, "acc@0.5": acc, "f1@0.5": f1}

def fold_metrics_from_scores(y_true, scores):
    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)
    yhat = (scores >= 0.0).astype(int)
    acc = accuracy_score(y_true, yhat)
    f1 = f1_score(y_true, yhat)
    return {"roc_auc": auc, "pr_auc": ap, "acc@0.0": acc, "f1@0.0": f1}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# --- 2. CARICAMENTO PROGRESSI ESISTENTI ---
if os.path.exists(checkpoint_file):
    print(f"🔄 Checkpoint trovato! Riprendo il lavoro da: {checkpoint_file}")
    with open(checkpoint_file, "rb") as f:
        checkpoint_data = pickle.load(f)
        results = checkpoint_data["results"]
        start_fold = checkpoint_data["last_completed_fold"] + 1
else:
    print("🚀 Nessun checkpoint trovato. Inizio la Cross-Validation da zero.")
    results = {
        "A_AngleVQC": [],
        "B_HamVQC": [],
        "C_HamKernelSVM": [],
    }
    start_fold = 1

# --- 3. CICLO DI CROSS-VALIDATION ---
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):

    # Salta i fold già completati (fondamentale per il secondo account!)
    if fold < start_fold:
        print(f"⏩ Fold {fold}/{N_FOLDS} già fatto. Salto...")
        continue

    print(f"\n===== ESECUZIONE: Fold {fold}/{N_FOLDS} =====")
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Preprocessing
    Xtr_ang, Xva_ang = fit_transform_fold(X_tr, X_va, n_qubits=n_qubits)

    # (A) AngleEmbedding VQC
    print(f"⏳ Fold {fold}: Training AngleVQC...")
    _, pA = train_vqc(vqc_angle, Xtr_ang, y_tr, Xva_ang, y_va, seed=100 + fold)
    mA = fold_metrics_from_probs(y_va, pA)
    results["A_AngleVQC"].append(mA)

    # (B) Hamiltonian embedding VQC
    print(f"⏳ Fold {fold}: Training HamVQC...")
    _, pB = train_vqc(vqc_hamiltonian, Xtr_ang, y_tr, Xva_ang, y_va, seed=200 + fold)
    mB = fold_metrics_from_probs(y_va, pB)
    results["B_HamVQC"].append(mB)

    # (C) Hamiltonian kernel SVM
    print(f"⏳ Fold {fold}: Calcolo matrici Kernel (Ottimizzato)...")
    K_tr = kernel_matrix(Xtr_ang, Xtr_ang)
    K_va = kernel_matrix(Xva_ang, Xtr_ang)

    svc = SVC(kernel="precomputed", C=SVC_C, class_weight="balanced")
    svc.fit(K_tr, y_tr)
    scores = svc.decision_function(K_va)

    mC = fold_metrics_from_scores(y_va, scores)
    results["C_HamKernelSVM"].append(mC)

    print(f"✅ Risultati Fold {fold} acquisiti.")

    # --- 4. SALVATAGGIO DEL CHECKPOINT DOPO OGNI FOLD ---
    checkpoint_data = {
        "results": results,
        "last_completed_fold": fold
    }
    with open(checkpoint_file, "wb") as f:
        pickle.dump(checkpoint_data, f)

    print(f"💾 PROGRESSO SALVATO: Fold {fold} è al sicuro su Drive.")

🚀 Nessun checkpoint trovato. Inizio la Cross-Validation da zero.

===== ESECUZIONE: Fold 1/5 =====
⏳ Fold 1: Training AngleVQC...


## 9) Summarize results (mean ± std)

In [ ]:
def summarize(results_list):
    dfm = pd.DataFrame(results_list)
    return dfm.mean().to_frame("mean").join(dfm.std().to_frame("std"))

summary_tables = {}
for model_name, res_list in results.items():
    summary_tables[model_name] = summarize(res_list)

summary = pd.concat(summary_tables, axis=0)
display(summary)

# Make a cleaner table for ROC AUC and PR AUC
rows = []
for model_name, res_list in results.items():
    dfm = pd.DataFrame(res_list)
    for metric in ["roc_auc", "pr_auc"]:
        rows.append({
            "model": model_name,
            "metric": metric,
            "mean": dfm[metric].mean(),
            "std": dfm[metric].std(),
        })

df_summary = pd.DataFrame(rows)
display(df_summary.pivot(index="model", columns="metric", values=["mean", "std"]))

## 10) Plot comparison (mean ROC AUC across folds)

In [ ]:
means = []
stds = []
labels = []

for model_name, res_list in results.items():
    dfm = pd.DataFrame(res_list)
    if "roc_auc" in dfm.columns:
        labels.append(model_name)
        means.append(dfm["roc_auc"].mean())
        stds.append(dfm["roc_auc"].std())

plt.figure()
plt.bar(labels, means, yerr=stds)
plt.xticks(rotation=20)
plt.ylabel("ROC AUC (mean ± std)")
plt.title("QML comparison on PIMA (cross-validation)")
plt.show()

## 11) Notes and research extensions
To turn this into a solid study:
- Run a **hyperparameter sweep** over: `n_qubits`, `N_LAYERS`, `EPOCHS`, `ALPHA_ZZ`, `T_EMBED`, `TROTTER_STEPS`
- Add **non-commuting Hamiltonian terms** (X, XX) and compare
- Replace PCA with supervised reduction (e.g., LDA) or learned classical embedding
- Use repeated CV or nested CV for fair model selection
- Add calibration curves and clinically relevant metrics (sensitivity/specificity)

If you want, I can generate a version that caches statevectors within folds to accelerate kernel computation.


In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt

print(f"💾 Salvataggio risultati in corso in: {drive_path}")

# 1. Salva Tabella completa (df_summary viene creato dal codice della Cross-Validation)
df_summary.to_csv(drive_path + "risultati_confronto.csv", index=False)

# 2. Creazione e salvataggio tabella pivot per la tesi
tabella_pivot = df_summary.pivot(index="model", columns="metric", values=["mean", "std"])
tabella_pivot.to_csv(drive_path + "tabella_tesi.csv")

# 3. Genera e Salva Grafico
plt.figure(figsize=(10, 6))
colori = ['#3498db', '#e74c3c', '#2ecc71']
# 'labels', 'means', 'stds' sono variabili create dal notebook durante la CV
plt.bar(labels, means, yerr=stds, capsize=5, color=colori[:len(labels)])
plt.xticks(rotation=20)
plt.ylabel("ROC AUC (mean ± std)")
plt.title(f"QML Comparison on PIMA ({n_qubits} Qubits - GPU)")
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.savefig(drive_path + "confronto_modelli_auc.png", dpi=300, bbox_inches='tight')
plt.show()

# 4. Salva dati grezzi (results)
with open(drive_path + "dati_completi_esperimento.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"✅ Tutto salvato correttamente nella cartella: Tesi_Risultati_{n_qubits}Qubits_CV")

In [ ]:
import matplotlib.pyplot as plt

# --- 1. INSERISCI QUI I TUOI DATI DAL DRIVE ---
qubits = [4, 6, 8]

# Sostituisci questi valori con gli AUC reali della Cross-Validation senza rumore
auc_angle = [0.55, 0.6384367575122292, 0.5851921733053808]  # Risultati AngleVQC
auc_ham = [0.50, 0.5764696016771488, 0.5486939203354297]    # Risultati HamVQC
auc_kernel = [0.56, 0.6302648497554157, 0.7295108315863033] # Risultati HamKernelSVM

# --- 2. CREAZIONE DEL GRAFICO ---
plt.figure(figsize=(10, 6))

# Creiamo le tre linee per i modelli quantistici
plt.plot(qubits, auc_angle, marker='o', linestyle='-', linewidth=2, label='AngleVQC')
plt.plot(qubits, auc_ham, marker='s', linestyle='-', linewidth=2, label='HamVQC')
plt.plot(qubits, auc_kernel, marker='^', linestyle='-', linewidth=2, label='HamKernelSVM')

# Linea tratteggiata per il benchmark classico (sostituisci 0.833 col tuo valore esatto se diverso)
plt.axhline(y=0.833, color='red', linestyle='--', label='Baseline Classica (Logistic Regression)')

# Formattazione
plt.title('Scalabilità dei Modelli Quantistici: AUC vs Numero di Qubit (Senza Rumore)', fontsize=14)
plt.xlabel('Numero di Qubit', fontsize=12)
plt.ylabel('ROC AUC', fontsize=12)
plt.xticks(qubits) # Forza l'asse X a mostrare solo 4, 6, 8
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='lower right', fontsize=11)

# Salvataggio su file
plt.savefig('grafico_scalabilita_qubit.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# @title
import matplotlib.pyplot as plt
import numpy as np

# Dati estratti dai tuoi file .pkl (Medie CV)
qubits = [4, 6, 8]
auc_angle = [0.520, 0.638, 0.585]
auc_ham = [0.489, 0.576, 0.549]
auc_kernel = [0.565, 0.630, 0.730]

# Deviazioni standard (Error Bars)
err_angle = [0.040, 0.095, 0.072]
err_ham = [0.053, 0.037, 0.048]
err_kernel = [0.036, 0.034, 0.022]

plt.figure(figsize=(10, 6))

# Linee dei modelli (Medie)
plt.errorbar(qubits, auc_angle, yerr=err_angle, marker='o', capsize=5, label='AngleVQC (Media CV)', linewidth=2)
plt.errorbar(qubits, auc_ham, yerr=err_ham, marker='s', capsize=5, label='HamVQC (Media CV)', linewidth=2)
plt.errorbar(qubits, auc_kernel, yerr=err_kernel, marker='^', capsize=5, label='HamKernelSVM (Media CV)', linewidth=2)

# Punti di picco della tesi (8 Qubit)
plt.scatter([8], [0.85], color='gold', s=120, edgecolors='black', zorder=5, label='HamVQC Peak (Thesis)')
plt.scatter([8], [0.84], color='silver', s=120, edgecolors='black', zorder=5, label='Kernel Peak (Thesis)')

# Baseline Classica
plt.axhline(y=0.833, color='red', linestyle='--', label='Baseline Classica (LogReg)')

plt.title('Scalabilità dei Modelli Quantistici: ROC AUC vs Numero di Qubit', fontsize=14)
plt.xlabel('Numero di Qubit', fontsize=12)
plt.ylabel('ROC AUC', fontsize=12)
plt.xticks(qubits)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=9)
plt.ylim(0.4, 0.9)

plt.savefig('grafico_scalabilita_finale.png', dpi=300, bbox_inches='tight')
plt.show()